# 04 · Offsets, commits, crash windows and replay
**Level:** intermediate · **Time:** 45 minutes · **Prerequisite:** 03.
You will restart a consumer under the same group, prove that commits mean the **next**
offset, and reproduce both duplicate processing and skipped work using real Kafka offsets.
One partition keeps the experiment unambiguous; production commits are per partition.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
require_broker()
topic = new_topic('offsets', partitions=1)
publish(topic, [{'number': i} for i in range(5)])


## 1. Process two records, then commit
Auto commit and auto offset storage are disabled. Commit the message only after the
handler succeeds. A consumer's fetch position can be ahead of its durable committed offset.


In [ ]:
from confluent_kafka import TopicPartition
group = f"lab.offsets.{RUN}"
c = consumer(group)
try:
    c.subscribe([topic])
    first = poll_records(c, count=2)
    for m in first:
        print("Processed", decode(m))
        c.commit(message=m, asynchronous=False)
    saved = c.committed([TopicPartition(topic, 0)], timeout=10)[0].offset
    print("Committed next offset:", saved)
    assert saved == first[-1].offset() + 1 == 2
finally:
    c.close()


## 2. Restart the same group
Despite `auto.offset.reset=earliest`, a valid committed offset takes priority.


In [ ]:
c = consumer(group)
try:
    c.subscribe([topic])
    rest = poll_records(c, count=3)
    assert [decode(m)["number"] for m in rest] == [2, 3, 4]
    for m in rest:
        c.commit(message=m, asynchronous=False)
    print("Resumed at", rest[0].offset())
finally:
    c.close()


## 3. Reproduce the at-least-once crash window
The business operation below is a list append. Close without committing to model a
crash **after** the side effect but **before** the offset commit. Closing is deterministic
here because auto commit is disabled; it is not a process-kill test.


In [ ]:
crash_group = f"lab.crash.{RUN}"
side_effects = []
c = consumer(crash_group)
try:
    c.subscribe([topic])
    m = poll_records(c, count=1)[0]
    side_effects.append(decode(m)["number"])
    print("Side effect completed; intentionally omit commit.")
finally:
    c.close()
c = consumer(crash_group)
try:
    c.subscribe([topic])
    m = poll_records(c, count=1)[0]
    side_effects.append(decode(m)["number"])
    c.commit(message=m, asynchronous=False)
finally:
    c.close()
assert side_effects == [0, 0]
print("Duplicate effect:", side_effects)


## 4. Reproduce the at-most-once crash window
Commit first, then omit the business operation. Restarting skips that operation.
This is why moving the commit earlier is not a general fix for duplicate effects.


In [ ]:
skip_group = f"lab.skip.{RUN}"
c = consumer(skip_group)
try:
    c.subscribe([topic])
    skipped = poll_records(c, count=1)[0]
    c.commit(message=skipped, asynchronous=False)
    # Deliberately no business operation after the commit.
finally:
    c.close()
c = consumer(skip_group)
try:
    c.subscribe([topic])
    next_message = poll_records(c, count=1)[0]
    assert decode(next_message)["number"] == 1
    print("Record 0's operation was skipped; restart received", decode(next_message))
finally:
    c.close()


## 5. Replay explicitly with manual assignment
`assign()` bypasses group subscription assignment. Use a separate replay group and do
not reset an active production group's offsets. `seek()` changes the current position,
not the persisted committed offset.


In [ ]:
c = consumer(f"lab.replay.{RUN}")
try:
    c.assign([TopicPartition(topic, 0, 0)])
    replay = poll_records(c, count=5)
    assert [decode(m)["number"] for m in replay] == list(range(5))
    c.seek(TopicPartition(topic, 0, 3))
    tail = poll_records(c, count=2)
    assert [decode(m)["number"] for m in tail] == [3, 4]
    print("Replay tail:", [decode(m) for m in tail])
finally:
    c.close()


## Exercise
Replace the list side effect with a dictionary keyed by event ID. Why is this insufficient
when the process restarts? What two writes must be in the same database transaction?
<details><summary>Solution hints</summary>
An in-memory dictionary disappears. Persist a unique processed-event ID and the business
effect atomically, then commit Kafka. On replay the unique ID suppresses the repeated effect.
Notebook 12 implements this in Django. Never commit a later message on a partition while
an earlier message on that partition remains unsuccessfully processed.
</details>
Next: **05 — groups and rebalancing**.
